# 04 · Shackleton: what the tethered explorer needs from the map

**Milestone 4 of the Lunar Hazard Map.** Shackleton is the first target of the team's tethered crater explorer. A solar Deploy Rover (DR) anchors on the rim. An Explore Rover (ER) rappels down a power-and-data tether, sets a charging Dock on the floor, and maps the floor in battery-powered trips. Shackleton sits inside NASA's 5 m map, so everything here uses the sharp map directly. The notebook answers five questions:

1. **Is the crater the size we think?** We check the measured diameter and depth against published values.
2. **Can a rover just drive in?** We find the gentlest possible drive from the rim to the floor, and how much of the floor a rover can drive at each slope limit.
3. **Where does the tether go down the wall?** We find the cheapest route from the DR site to the floor and report its true length, steepest point and share in shadow.
4. **Where should the Dock go?** We try candidate Dock sites on the floor and keep the one the ER can reach the most floor from, by *driving* distance. That also sets how much tether the ER lays across the floor.
5. **Can the ER radio the Dock?** We check line of sight from the reachable floor to the Dock, including the Moon's curvature.

It ends with a table comparing the design doc's numbers to what the map measures, and writes them to `results/m4_summary.json`, which `concept-sizing/sizing.py` reads.

**Before running:** you need `ldem_87s_5mpp.tif` and `ldec_87s_5mpp.tif` (Milestone 1) and `LPSR_80S_20MPP_ADJ.TIF` (Milestone 2), all in `~/lunar/data/`. There's nothing new to download. It takes about 5–10 minutes (estimated), mostly the route search and the Dock trade.

**Number labels.** *Sourced* = from a cited page or paper. *Placeholder* = a setting the team still has to confirm. *Measured* = computed here from NASA data.


**Data citations:** Barker et al. 2021, *Planet. Space Sci.* 203, 105119, doi:10.1016/j.pss.2020.105119 (5 m mosaic). Barker et al. 2023, *Planet. Sci. J.* 4, 183, doi:10.3847/PSJ/acf3e1 (permanent shadow map).

## Setup

In [ ]:
import os, json, time, math
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.warp import reproject, Resampling
from affine import Affine
from scipy import ndimage
from skimage.graph import MCP_Geometric

plt.rcParams.update({"figure.dpi": 110, "image.origin": "upper"})
pd.set_option("display.width", 160)

In [ ]:
# ---------------- Settings ----------------
DATA = Path(os.environ.get("LUNAR_DATA", Path.home() / "lunar" / "data"))
FIG = Path("figures"); RESULTS = Path("results")
for d in (FIG, RESULTS):
    d.mkdir(parents=True, exist_ok=True)
F5 = {"elev": DATA / "ldem_87s_5mpp.tif", "count": DATA / "ldec_87s_5mpp.tif"}
F_PSR = DATA / "LPSR_80S_20MPP_ADJ.TIF"

MOON_R = 1_737_400.0                          # m, LOLA reference sphere
SHACK_LAT, SHACK_LON = -89.67, 129.78         # SOURCED: USGS Gazetteer of Planetary Nomenclature (as notebooks 01-02)
PUB_DIAMETER_KM, PUB_DEPTH_KM = 21.0, 4.2     # SOURCED: published Shackleton diameter and depth (see Sources)
CROP_KM = float(os.environ.get("LUNAR_CROP_KM", 30))   # window around the crater centre (setting)
GRID_M = 10                                   # analysis grid. Each 10 m cell takes the STEEPEST of its four 5 m
                                              # pixels, so anything that avoids a 10 m no-go cell also avoids 5 m no-go.

# Terrain classes (same as notebooks 01-03)
SAFE_MAX_DEG = 15.0     # SOURCED: NASA VIPER is rated for slopes up to 15 deg
NOGO_MIN_DEG = 25.0     # PLACEHOLDER: project plan

# Tethered explorer settings (numbers marked "concept-sizing" come from ../concept-sizing/sizing.py)
ER_MAX_DEG = 15.0       # PLACEHOLDER: Explore Rover free-driving limit, set to VIPER's sourced 15 deg rating
RAPPEL_MAX_DEG = 45.0   # SOURCED: Michigan Tech T-REX tether rover was tested on 45 deg simulant slopes
SUNLIT_PENALTY = 3.0    # PLACEHOLDER: a sunlit metre of tether "costs" 3 shadowed metres. Warm aluminium has about
                        # 4x the resistance of shadowed aluminium (concept-sizing assumption), and sunlit cable cycles hot/cold.
SLOPE_WEIGHT = 2.0      # setting: extra route cost for steepness (cost = 1 + 2 x (slope / 45)^2 per metre)
FLOOR_FRAC = 0.05       # PLACEHOLDER: "floor" = the lowest 5% of the crater's depth, joined to the lowest point
DR_XY = None            # None = PLACEHOLDER rule: the highest point on the rim crest. Or set (x, y) in metres,
                        # ideally a site picked from NASA illumination maps.
ER_REACH_M = {          # one-way trip length the ER can drive and still get back to the Dock (concept-sizing, range equation)
    "baseline (0.10 m/s)": 2486.0,
    "heater units (0.10 m/s)": 3244.0,
    "faster (0.15 m/s)": 3611.0,
}
BASE_REACH = "baseline (0.10 m/s)"
N_DOCK_CANDIDATES = 60  # setting: candidate Dock sites tried, spread evenly over the drivable floor
TETHER_SLACK = 0.10     # PLACEHOLDER: extra tether for route wiggles, same as concept-sizing
ER_ANT_M = 1.0          # PLACEHOLDER: ER radio antenna height
DOCK_ANT_M = 1.5        # PLACEHOLDER: Dock radio antenna height
DESIGN_DOC = dict(      # numbers in the design doc (Tethered Crater Explorer - Baseline Design, Nov 20 2026)
    wall_km=9.4, floor_run_km=3.0, tether_km=13.6, reach_pct=67.0, reach_pct_heater=100.0,
)

missing = [p.name for p in [*F5.values(), F_PSR] if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing in {DATA}: {missing}. See download_m1.sh / download_m2.sh.")
print("Data folder:", DATA)

In [ ]:
# ---------------- Helpers ----------------
def latlon_to_xy(lat, lon):
    rho = 2 * MOON_R * np.tan(np.radians(90 + np.asarray(lat)) / 2)
    return rho * np.sin(np.radians(lon)), rho * np.cos(np.radians(lon))

def read_window(path, w):
    with rasterio.open(path) as s:
        a = s.read(1, window=w, masked=True)
    a = np.ma.filled(a.astype("float32"), np.nan)
    a[~np.isfinite(a) | (np.abs(a) > 1e30)] = np.nan
    return a

def slope_deg(dem, res):
    gy, gx = np.gradient(dem, res)
    return np.degrees(np.arctan(np.hypot(gx, gy))).astype("float32")

def hillshade(dem, res, az=45, alt=45):
    gy, gx = np.gradient(dem, res)
    sl = np.arctan(np.hypot(gx, gy)); asp = np.arctan2(-gy, -gx)
    a, z = np.radians(360 - az + 90), np.radians(90 - alt)
    return np.clip(np.cos(z) * np.cos(sl) + np.sin(z) * np.sin(sl) * np.cos(a - asp), 0, 1)

def block(a, k, f):
    n0, n1 = a.shape[0] // k, a.shape[1] // k
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return f(a[:n0 * k, :n1 * k].reshape(n0, k, n1, k), axis=(1, 3))

CLASS_CMAP = ListedColormap(["#2e9e5b", "#f0b429", "#d64545"])
def classify(s):
    c = np.full(s.shape, 255, np.uint8); ok = np.isfinite(s)
    c[ok & (s < SAFE_MAX_DEG)] = 0; c[ok & (s >= SAFE_MAX_DEG) & (s < NOGO_MIN_DEG)] = 1; c[ok & (s >= NOGO_MIN_DEG)] = 2
    return c

def km(v): return v / 1000.0

## A · Load Shackleton and check its shape

**Plain English.** We cut a square around the crater out of the 5 m map. Then we find the rim (the highest point along each of 360 directions from the centre) and the floor (the lowest part). We check the diameter and depth against published numbers. If they match, the map and our geometry are trustworthy.

In [ ]:
t0 = time.time()
CX, CY = latlon_to_xy(SHACK_LAT, SHACK_LON)
half = CROP_KM * 1000 / 2
with rasterio.open(F5["elev"]) as s:
    w = from_bounds(CX - half, CY - half, CX + half, CY + half, transform=s.transform)
    w = Window(int(round(w.col_off)), int(round(w.row_off)), int(round(w.width)) // 2 * 2, int(round(w.height)) // 2 * 2)
    t5, CRS5 = s.window_transform(w), s.crs
    assert abs(s.transform.a - 5) < 1e-6
e5 = read_window(F5["elev"], w)
c5 = read_window(F5["count"], w)
s5 = slope_deg(e5, 5)
print(f"5 m window: {e5.shape[1]} x {e5.shape[0]} px ({CROP_KM:.0f} km square), read in {time.time() - t0:.0f} s, "
      f"no-data share {np.isnan(e5).mean():.2%}")

# 10 m analysis grid
K = GRID_M // 5
e10 = block(e5, K, np.nanmean).astype("float32")
s10 = block(s5, K, np.nanmax).astype("float32")             # steepest 5 m pixel in each 10 m cell
real10 = block((np.nan_to_num(c5) > 0).astype("float32"), K, np.mean)
t10 = t5 * Affine.scale(K)
valid10 = np.isfinite(e10) & np.isfinite(s10)

# Permanent shadow map (20 m) onto the 10 m grid
with rasterio.open(F_PSR) as s:
    b = rasterio.transform.array_bounds(e10.shape[0], e10.shape[1], t10)
    wp = from_bounds(b[0] - 100, b[1] - 100, b[2] + 100, b[3] + 100, transform=s.transform).round_offsets().round_lengths()
    psr_src = s.read(1, window=wp, masked=True).filled(0).astype("float32"); tp = s.window_transform(wp)
psr10 = np.zeros(e10.shape, "float32")
reproject(psr_src, psr10, src_transform=tp, src_crs=CRS5, dst_transform=t10, dst_crs=CRS5, resampling=Resampling.nearest)
psr10 = psr10 > 0

# Coordinates of 10 m cell centres, distance from crater centre
rows, cols = np.indices(e10.shape)
X10 = t10.c + (cols + 0.5) * t10.a
Y10 = t10.f + (rows + 0.5) * t10.e
R10 = np.hypot(X10 - CX, Y10 - CY)
def rc_of(x, y):
    return int((y - t10.f) / t10.e), int((x - t10.c) / t10.a)
def xy_of(r, c):
    return t10.c + (c + 0.5) * t10.a, t10.f + (r + 0.5) * t10.e

# Rim crest: highest point along 360 rays from the centre, between 4 km and the window edge
radii = np.arange(4000, half - 200, GRID_M)
az = np.radians(np.arange(360))
RX = CX + np.outer(np.sin(az), radii); RY = CY + np.outer(np.cos(az), radii)
rr_ = ((RY - t10.f) / t10.e).astype(int); cc_ = ((RX - t10.c) / t10.a).astype(int)
prof = np.where(np.isfinite(e10[rr_, cc_]), e10[rr_, cc_], -np.inf)
k = prof.argmax(axis=1)
rim_r, rim_z = radii[k], prof[np.arange(360), k]
rim_rc = np.stack([rr_[np.arange(360), k], cc_[np.arange(360), k]], 1)
RIM_R = float(np.median(rim_r)); RIM_Z = float(np.median(rim_z))

# Floor: lowest part of the bowl, joined to the lowest point
inner = valid10 & (R10 < 0.6 * RIM_R)
z_floor = float(np.nanpercentile(e10[inner], 0.5))
DEPTH = RIM_Z - z_floor
lab, _ = ndimage.label(valid10 & (R10 < RIM_R) & (e10 < z_floor + FLOOR_FRAC * DEPTH))
low = np.unravel_index(np.where(inner, e10, np.inf).argmin(), e10.shape)
floor = lab == lab[low] if lab[low] > 0 else np.zeros_like(valid10)
floor_area = floor.sum() * GRID_M ** 2
wall = valid10 & (R10 < RIM_R) & ~floor & ~ndimage.binary_dilation(floor, iterations=3)

# Deploy Rover (DR) site: the top of the tether
if DR_XY is None:
    i = int(np.argmax(rim_z)); DR_RC = tuple(rim_rc[i]); DR_RULE = "placeholder: highest point on the rim crest"
else:
    DR_RC = rc_of(*DR_XY); DR_RULE = "set by the team"
dr_xy = xy_of(*DR_RC)

geo = pd.DataFrame([
    dict(item="Rim diameter (km)", measured=2 * km(RIM_R), published=PUB_DIAMETER_KM, note="median over 360 directions"),
    dict(item="Depth, rim to floor (km)", measured=km(DEPTH), published=PUB_DEPTH_KM, note="median rim height minus floor"),
    dict(item="Floor, equal-area diameter (km)", measured=2 * km(math.sqrt(floor_area / math.pi)), published=np.nan,
         note=f"lowest {FLOOR_FRAC:.0%} of depth (placeholder)"),
    dict(item="Median wall slope, 5 m (deg)", measured=float(np.nanmedian(s5[np.repeat(np.repeat(wall, K, 0), K, 1)[:s5.shape[0], :s5.shape[1]]])),
         published=np.nan, note="between floor and rim"),
    dict(item="Floor in permanent shadow (%)", measured=100 * psr10[floor].mean(), published=np.nan, note="NASA PSR map"),
    dict(item="Floor pixels with a real laser shot (%)", measured=100 * float(real10[floor].mean()), published=np.nan,
         note="rest is filled in"),
])
print("Shackleton geometry (measured vs SOURCED published values):")
display(geo.round(2))
print(f"DR site ({DR_RULE}): x {km(dr_xy[0]):.2f} km, y {km(dr_xy[1]):.2f} km, "
      f"elevation {e10[DR_RC]:.0f} m, in permanent shadow: {bool(psr10[DR_RC])}")

In [ ]:
step = 2
ext10 = [km(t10.c), km(t10.c + t10.a * e10.shape[1]), km(t10.f + t10.e * e10.shape[0]), km(t10.f)]
hs = hillshade(e10[::step, ::step], GRID_M * step)
def base_map(ax, title):
    ax.imshow(hs, cmap="gray", extent=ext10)
    ax.contour(X10[::step, ::step] / 1e3, Y10[::step, ::step] / 1e3, psr10[::step, ::step].astype(float), levels=[0.5],
               colors="#4aa3ff", linewidths=0.6)
    ax.plot(*(np.array([xy_of(*rc) for rc in rim_rc]).T / 1e3), ".", ms=1.5, color="#ffd166")
    ax.contour(X10[::step, ::step] / 1e3, Y10[::step, ::step] / 1e3, floor[::step, ::step].astype(float), levels=[0.5],
               colors="white", linewidths=1)
    ax.plot(km(dr_xy[0]), km(dr_xy[1]), "^", ms=11, color="#ffd166", mec="k")
    ax.set_title(title, fontsize=10); ax.set_xlabel("x (km, polar stereographic)"); ax.set_ylabel("y (km)")
fig, ax = plt.subplots(figsize=(7.5, 7.5))
base_map(ax, "Shackleton: rim crest (yellow dots), floor (white), permanent shadow (blue), DR site (triangle)")
plt.tight_layout(); plt.savefig(FIG / "M4_shackleton_geometry.png", dpi=150); plt.show()

## B · The hazard map

The safe / caution / no-go classes from the 5 m slope, shown on the 10 m grid (each cell takes its steepest 5 m pixel, so this map is slightly cautious).

In [ ]:
cls10 = classify(s10)
zones = {"floor": floor, "wall": wall, "rim zone (rim to rim + 2 km)": valid10 & (R10 >= RIM_R) & (R10 < RIM_R + 2000)}
cls5 = classify(s5)
def up(m): return np.repeat(np.repeat(m, K, 0), K, 1)[:cls5.shape[0], :cls5.shape[1]]
rowsB = []
for name, m in zones.items():
    c = cls5[up(m)]; c = c[c != 255]
    rowsB.append(dict(zone=name, area_km2=m.sum() * GRID_M ** 2 / 1e6,
                      **{f"{n} %": 100 * (c == i).mean() for i, n in enumerate(["safe", "caution", "no-go"])}))
B = pd.DataFrame(rowsB)
print("Hazard classes from 5 m slope (measured):"); display(B.round(1))

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.imshow(hs, cmap="gray", extent=ext10)
ax.imshow(np.ma.masked_equal(cls10[::step, ::step], 255), cmap=CLASS_CMAP, vmin=-0.5, vmax=2.5, alpha=0.55,
          extent=ext10, interpolation="nearest")
ax.contour(X10[::step, ::step] / 1e3, Y10[::step, ::step] / 1e3, floor[::step, ::step].astype(float), levels=[0.5], colors="white", linewidths=1)
ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(["#2e9e5b", "#f0b429", "#d64545"],
          [f"safe < {SAFE_MAX_DEG:.0f}°", "caution", f"no-go ≥ {NOGO_MIN_DEG:.0f}°"])], loc="lower left", fontsize=8)
ax.set_title("Shackleton hazard map (5 m slope, shown at 10 m)", fontsize=10)
ax.set_xlabel("x (km)"); ax.set_ylabel("y (km)")
plt.tight_layout(); plt.savefig(FIG / "M4_hazard_map.png", dpi=150); plt.show()

## C · Can a rover just drive in?

**Plain English.** Two numbers set the rover's design:

- **Drive-down slope:** the gentlest possible drive from the rim to the floor. Every path down has a steepest spot. This finds the path whose steepest spot is as gentle as possible. A rover rated below this number *cannot* drive into the crater, so something has to lower it. That is why the ER rappels on the tether.
- **Floor reach:** at each slope limit, how much of the floor a rover can drive on, and how much of that is one connected piece it can roam without help.

In [ ]:
ok = valid10
rim_mask = np.zeros_like(ok); rim_mask[rim_rc[:, 0], rim_rc[:, 1]] = True
def connected(limit, src_mask):
    lab, _ = ndimage.label(ok & (s10 < limit), structure=np.ones((3, 3)))
    ids = np.unique(lab[src_mask & (lab > 0)])
    return np.isin(lab, ids[ids > 0])
def bottleneck(src_mask, lo=0.0, hi=90.0):
    if not connected(hi, src_mask)[floor].any():
        return np.nan
    while hi - lo > 0.25:
        mid = (lo + hi) / 2
        if connected(mid, src_mask)[floor].any(): hi = mid
        else: lo = mid
    return hi
dr_mask = np.zeros_like(ok); dr_mask[DR_RC] = True
DRIVE_DOWN_DR = bottleneck(dr_mask)
DRIVE_DOWN_ANY = bottleneck(rim_mask)
print(f"Gentlest drive from the DR SITE to the floor needs a rover rated for {DRIVE_DOWN_DR:.1f} deg (measured).")
print(f"Gentlest drive from ANYWHERE on the rim crest needs {DRIVE_DOWN_ANY:.1f} deg (measured).")

limits = [10, 15, 20, 25, 30, 35, 40, 45]
rowsC = []
for L in limits:
    drv = floor & ok & (s10 < L)
    lab, n = ndimage.label(drv, structure=np.ones((3, 3)))
    biggest = np.bincount(lab.ravel())[1:].max() if n else 0
    rowsC.append(dict(slope_limit_deg=L, floor_drivable_pct=100 * drv.sum() / floor.sum(),
                      largest_connected_floor_pct=100 * biggest / floor.sum(),
                      can_drive_rim_to_floor=bool(np.isfinite(DRIVE_DOWN_ANY) and L >= DRIVE_DOWN_ANY)))
C = pd.DataFrame(rowsC)
print("\nFloor reach by rover slope limit (measured):"); display(C.round(1))

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(C.slope_limit_deg, C.floor_drivable_pct, "o-", label="floor a rover can drive on")
ax.plot(C.slope_limit_deg, C.largest_connected_floor_pct, "s--", label="largest connected piece")
if np.isfinite(DRIVE_DOWN_ANY):
    ax.axvline(DRIVE_DOWN_ANY, color="#d64545", ls=":", label=f"needed to drive down: {DRIVE_DOWN_ANY:.0f}°")
ax.axvline(SAFE_MAX_DEG, color="gray", lw=1, label="VIPER rating 15°")
ax.set_xlabel("rover slope limit (deg)"); ax.set_ylabel("% of Shackleton floor"); ax.set_ylim(0, 102)
ax.set_title("How capable must the rover be? (measured)", fontsize=10); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / "M4_rover_requirement.png", dpi=150); plt.show()

## D · The tether route down the wall

**Plain English.** The ER starts at the DR site and rappels down to the floor, paying tether out from its own spool. The search looks for the cheapest path, where:

- each metre costs more the steeper it is,
- a metre outside permanent shadow costs 3× more (placeholder), because warm aluminium carries power less well and sunlit cable heats and cools every day,
- anything steeper than 45° is off limits (the slope the T-REX tether rover has been tested on).

For comparison, it also finds the route with no preference for shadow. Lengths are true 3D lengths along the ground, not map distance. On the way back up, the ER reels in along this same path, so the tether never gets dragged across new ground.

In [ ]:
def tether_cost(sun_penalty):
    c = (1 + SLOPE_WEIGHT * (s10 / RAPPEL_MAX_DEG) ** 2) * np.where(psr10, 1.0, sun_penalty)
    return np.where(ok & (s10 <= RAPPEL_MAX_DEG), c, np.inf).astype("float64")

def plan(sun_penalty):
    t0 = time.time()
    m = MCP_Geometric(tether_cost(sun_penalty))
    cum, _ = m.find_costs([DR_RC])
    tgt = np.where(floor & np.isfinite(cum), cum, np.inf)
    if not np.isfinite(tgt).any():
        return None
    end = np.unravel_index(tgt.argmin(), tgt.shape)
    path = np.array(m.traceback(end))
    print(f"  route search (sunlit penalty {sun_penalty}) took {time.time() - t0:.0f} s")
    return path

def route_stats(path, name):
    r, c = path[:, 0], path[:, 1]
    dxy = np.hypot(np.diff(r), np.diff(c)) * GRID_M
    dz = np.diff(e10[r, c].astype("float64"))
    seg3d = np.hypot(dxy, np.nan_to_num(dz))
    on_floor = floor[r, c]
    first_floor = int(np.argmax(on_floor)) if on_floor.any() else len(r) - 1
    shadow = psr10[r, c][1:]
    return dict(route=name, length_3d_km=km(seg3d.sum()), map_length_km=km(dxy.sum()),
                wall_part_3d_km=km(seg3d[:first_floor].sum()), drop_m=float(e10[r[0], c[0]] - e10[r[-1], c[-1]]),
                in_shadow_pct=100 * float((seg3d * shadow).sum() / seg3d.sum()),
                steepest_5m_deg=float(s10[r, c].max()), mean_slope_deg=float(np.degrees(np.arctan(
                    np.abs(dz).sum() / max(dxy.sum(), 1)))))

route = plan(SUNLIT_PENALTY)
route_plain = plan(1.0)
if route is None:
    raise RuntimeError("No route from the DR site to the floor under the 45 deg limit. Try another DR_XY.")
D = pd.DataFrame([route_stats(route, f"prefers shadow (x{SUNLIT_PENALTY:g})"),
                  route_stats(route_plain, "no shadow preference")])
print("Tether routes, DR site to floor (measured):"); display(D.round(2))
TETHER_END = tuple(route[-1])   # where the wall route first reaches the floor

# Profile
r, c = route[:, 0], route[:, 1]
dist = np.concatenate([[0], np.cumsum(np.hypot(np.hypot(np.diff(r), np.diff(c)) * GRID_M, np.diff(e10[r, c])))]) / 1e3
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(dist, e10[r, c], color="k", lw=1.2)
ax.fill_between(dist, e10[r, c].min() - 100, e10[r, c], where=psr10[r, c], color="#4aa3ff", alpha=0.3, step="mid",
                label="permanent shadow")
ax.set_xlabel("distance along the ground from the DR site (km)"); ax.set_ylabel("elevation (m)")
ax.set_title("Tether route profile, DR site to floor (measured)", fontsize=10); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / "M4_tether_profile.png", dpi=150); plt.show()

## E · Where should the Dock go?

**Plain English.** The ER can only drive so far from the Dock and still get back on one battery: about 2.5 km of driving in the baseline (from the range equation in `concept-sizing`). So the Dock's spot decides how much of the floor gets mapped.

1. The wall route reaches the floor at its edge. The ER then lays tether across the floor to the Dock, so a Dock deeper in costs more tether.
2. We spread candidate Dock sites over the floor the ER can drive (under its slope limit), and for each one measure the **driving distance** to every floor cell. That is the real path around no-go spots, not a straight line.
3. We keep the site that puts the most floor within reach. Among near-ties (within 1 point of the best), we keep the one with the shortest tether run.

The scatter plot is the trade: each dot is a candidate Dock, showing tether laid across the floor against floor in reach. A Dock too close to the wall wastes reach on the wall side; one too far away costs tether.

In [ ]:
drv_floor = floor & ok & (s10 < ER_MAX_DEG)
if not drv_floor.any():
    raise RuntimeError(f"No floor is drivable under {ER_MAX_DEG} deg; raise ER_MAX_DEG.")
# Main drivable floor = the biggest connected piece the ER can roam
lab_f, n_f = ndimage.label(drv_floor, structure=np.ones((3, 3)))
main_floor = lab_f == (np.bincount(lab_f.ravel())[1:].argmax() + 1)
dd = ndimage.distance_transform_edt(~main_floor, return_indices=True)[1]
ENTRY = (int(dd[0][TETHER_END]), int(dd[1][TETHER_END]))     # nearest main-floor cell to where the tether lands
ENTRY_LINK_M = math.hypot(ENTRY[0] - TETHER_END[0], ENTRY[1] - TETHER_END[1]) * GRID_M

rs, cs = np.nonzero(main_floor); r0, r1, c0, c1 = rs.min() - 2, rs.max() + 3, cs.min() - 2, cs.max() + 3
sub = main_floor[r0:r1, c0:c1]
mcp_floor = MCP_Geometric(np.where(sub, 1.0, np.inf))
def drive_from(rc):
    d, _ = mcp_floor.find_costs([(rc[0] - r0, rc[1] - c0)])
    return d * GRID_M                      # metres of driving, on the sub-grid
d_entry = drive_from(ENTRY)

# Candidate Dock sites: evenly spread over the main floor
t0 = time.time()
gap = max(int(math.sqrt(sub.sum() / N_DOCK_CANDIDATES)), 1)   # grid spacing (cells) giving about N candidates
gr, gc = np.nonzero(sub & (np.indices(sub.shape)[0] % gap == gap // 2) & (np.indices(sub.shape)[1] % gap == gap // 2))
cands = [(int(a) + r0, int(b) + c0) for a, b in zip(gr, gc)]
rowsE = []
for rc in cands:
    d = drive_from(rc)
    rowsE.append(dict(x_km=km(xy_of(*rc)[0]), y_km=km(xy_of(*rc)[1]), rc=rc,
                      floor_run_km=km(d_entry[rc[0] - r0, rc[1] - c0]),
                      reach_pct_floor=100 * (d[sub] <= ER_REACH_M[BASE_REACH]).sum() / floor.sum(),
                      mean_drive_km=km(d[sub].mean()), worst_drive_km=km(d[sub].max())))
E_all = pd.DataFrame(rowsE)
best = E_all.reach_pct_floor.max()
E_pick = E_all[E_all.reach_pct_floor >= best - 1.0].sort_values("floor_run_km").iloc[0]
DOCK = tuple(E_pick.rc)
d_dock = drive_from(DOCK)
print(f"Tried {len(cands)} Dock sites in {time.time() - t0:.0f} s.")

# Tether laid across the floor = the ER's drive from the floor entry to the Dock
mcp_floor.find_costs([(ENTRY[0] - r0, ENTRY[1] - c0)])
floor_path = np.array(mcp_floor.traceback((DOCK[0] - r0, DOCK[1] - c0))) + [r0, c0]
FLOOR_RUN_KM = km(d_entry[DOCK[0] - r0, DOCK[1] - c0])
WALL_KM = float(D.iloc[0].length_3d_km)
TETHER_KM = (WALL_KM + km(ENTRY_LINK_M) + FLOOR_RUN_KM) * (1 + TETHER_SLACK)

rowsR = []
for name, R in ER_REACH_M.items():
    rowsR.append(dict(case=name, reach_km=km(R),
                      floor_in_reach_pct=100 * (d_dock[sub] <= R).sum() / floor.sum(),
                      drivable_floor_in_reach_pct=100 * (d_dock[sub] <= R).mean()))
E = pd.DataFrame(rowsR)
print(f"Drivable floor (< {ER_MAX_DEG:.0f} deg, placeholder): {100 * drv_floor.sum() / floor.sum():.1f}% of the floor; "
      f"the main connected piece is {100 * sub.sum() / floor.sum():.1f}% (measured).")
print(f"Chosen Dock: x {E_pick.x_km:.2f} km, y {E_pick.y_km:.2f} km. Tether across the floor {FLOOR_RUN_KM:.2f} km; "
      f"worst drive to the Dock {E_pick.worst_drive_km:.2f} km (measured).")
print(f"Tether total: wall {WALL_KM:.2f} + entry link {km(ENTRY_LINK_M):.2f} + floor {FLOOR_RUN_KM:.2f} km, "
      f"+{TETHER_SLACK:.0%} slack = {TETHER_KM:.2f} km")
print("Floor in reach of the Dock, by driving distance (measured; reach values from concept-sizing):"); display(E.round(1))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.scatter(E_all.floor_run_km, E_all.reach_pct_floor, s=18, color="#8a94a6", label="candidate Dock sites")
ax.scatter([E_pick.floor_run_km], [E_pick.reach_pct_floor], s=70, color="#3b6fd8", zorder=3, label="chosen Dock")
ax.set_xlabel("tether laid across the floor (km)"); ax.set_ylabel(f"% of floor within {km(ER_REACH_M[BASE_REACH]):.1f} km drive")
ax.set_ylim(0, 102); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower right")
ax.set_title("Dock trade: floor in reach vs tether laid across the floor (measured)", fontsize=10)
plt.tight_layout(); plt.savefig(FIG / "M4_dock_trade.png", dpi=150); plt.show()

## F · Can the ER radio the Dock?

**Plain English.** The ER uploads data through the Dock and the tether. When it can see the Dock by radio, it can also report in the middle of a trip. When it can't, it stores data and uploads at the next charge. We draw a straight line from the ER's antenna (placeholder 1 m up) to the Dock's antenna (placeholder 1.5 m up). If the ground pokes above that line anywhere, the link is blocked. The Moon's curve is included: across 5 km the ground bulges about 2 m above a straight line.

We check 1,500 random points on the floor within the ER's baseline reach.

In [ ]:
def visible(rc_from, h_from, rc_to, h_to, step_m=5.0):
    x0, y0 = xy_of(*rc_from); x1, y1 = xy_of(*rc_to)
    Dd = math.hypot(x1 - x0, y1 - y0); n = max(int(Dd / step_m), 2)
    f = np.linspace(0, 1, n)[2:-2]
    xs, ys = x0 + f * (x1 - x0), y0 + f * (y1 - y0)
    rr = (ys - t10.f) / t10.e - 0.5; cc = (xs - t10.c) / t10.a - 0.5
    ground = ndimage.map_coordinates(np.nan_to_num(e10, nan=-1e9), [rr, cc], order=1, mode="nearest")
    z0 = e10[rc_from] + h_from; z1 = e10[rc_to] + h_to
    line = z0 + f * (z1 - z0) - (f * Dd) * ((1 - f) * Dd) / (2 * MOON_R)   # straight line sags below a curved surface
    return bool(np.all(line > ground)), float((line - ground).min())

inreach = np.zeros_like(floor)
inreach[r0:r1, c0:c1] = sub & (d_dock <= ER_REACH_M[BASE_REACH])
fr, fc = np.nonzero(inreach)
pick = np.random.default_rng(23).choice(len(fr), size=min(1500, len(fr)), replace=False)   # random sample of reachable floor
los = np.array([visible((fr[i], fc[i]), ER_ANT_M, DOCK, DOCK_ANT_M) for i in pick], dtype=object)
seen = np.array([v[0] for v in los], bool)
RADIO_LOS_PCT = 100 * seen.mean()
print(f"Reachable floor with direct radio to the Dock: {RADIO_LOS_PCT:.0f}% "
      f"({len(pick):,} sample points; antenna heights are placeholders; measured)")
los_xy = np.array([xy_of(fr[i], fc[i]) for i in pick]) / 1e3

## G · The explorer map

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
base_map(ax, "Tethered explorer at Shackleton (measured from NASA LOLA 5 m map)")
ax.imshow(np.ma.masked_where(~(cls10[::step, ::step] == 2), cls10[::step, ::step]), cmap=ListedColormap(["#d64545"]),
          alpha=0.35, extent=ext10, interpolation="nearest")
for path, col, ls in ((route_plain, "#bbbbbb", "--"), (route, "#3b6fd8", "-"), (floor_path, "#3b6fd8", "-")):
    xs, ys = zip(*[xy_of(r_, c_) for r_, c_ in path])
    ax.plot(np.array(xs) / 1e3, np.array(ys) / 1e3, ls, color=col, lw=2.2)
full = np.full(floor.shape, np.nan); full[r0:r1, c0:c1] = np.where(sub, d_dock, np.nan)
reach_colors = ["#2e9e5b", "#f0b429", "#c77dff"]
for (name, R), col in zip(ER_REACH_M.items(), reach_colors):
    ax.contour(X10[::step, ::step] / 1e3, Y10[::step, ::step] / 1e3, np.ma.masked_invalid(full[::step, ::step]),
               levels=[R], colors=col, linewidths=1.4)
ax.scatter(los_xy[~seen, 0], los_xy[~seen, 1], s=3, color="#ff6b6b", zorder=3)
dx, dy = xy_of(*DOCK)
ax.plot(km(dx), km(dy), "s", ms=11, color="#3b6fd8", mec="w", zorder=4)
ax.legend(handles=[Line2D([], [], color="#3b6fd8", lw=2.2, label="tether, prefers shadow (wall + floor run)"),
                   Line2D([], [], color="#bbbbbb", lw=2.2, ls="--", label="wall route, no shadow preference"),
                   Line2D([], [], marker="s", ls="", color="#3b6fd8", mec="w", ms=10, label="Dock (chosen site)"),
                   *[Line2D([], [], color=c, lw=1.4, label=f"ER reach, {n}") for n, c in zip(ER_REACH_M, reach_colors)],
                   Line2D([], [], marker=".", ls="", color="#ff6b6b", label="no direct radio to Dock"),
                   Line2D([], [], color="#4aa3ff", lw=1, label="permanent shadow edge"),
                   Line2D([], [], color="white", lw=1, label="floor"),
                   Line2D([], [], marker="^", ls="", color="#ffd166", mec="k", ms=10, label="DR site (placeholder)"),
                   Patch(color="#d64545", alpha=0.35, label=f"no-go ≥ {NOGO_MIN_DEG:.0f}°")],
          loc="lower left", fontsize=7.5, framealpha=0.85)
plt.tight_layout(); plt.savefig(FIG / "M4_explorer_map.png", dpi=170); plt.show()

## H · Design doc numbers vs the map

The design doc's numbers next to what the map measures. The reach numbers in the doc came from a straight-line estimate, so the driving-distance numbers here replace them. Where the two disagree a lot, update the design doc before the proposal.

In [ ]:
base = E.set_index("case").loc[BASE_REACH]
heat = E.set_index("case").loc["heater units (0.10 m/s)"]
H = pd.DataFrame([
    dict(item="Crater diameter (km)", design_doc="about 21 (sourced)", measured=round(2 * km(RIM_R), 1)),
    dict(item="Tether down the wall (km)", design_doc=DESIGN_DOC["wall_km"], measured=round(WALL_KM, 2)),
    dict(item="Tether across the floor to the Dock (km)", design_doc=f"{DESIGN_DOC['floor_run_km']} (placeholder)",
         measured=round(km(ENTRY_LINK_M) + FLOOR_RUN_KM, 2)),
    dict(item=f"Total tether incl. {TETHER_SLACK:.0%} slack (km)", design_doc=DESIGN_DOC["tether_km"], measured=round(TETHER_KM, 2)),
    dict(item="Steepest point on the tether route (deg)", design_doc=41, measured=round(float(D.iloc[0].steepest_5m_deg), 1)),
    dict(item="Tether route in permanent shadow (%)", design_doc=69, measured=round(float(D.iloc[0].in_shadow_pct), 0)),
    dict(item="Slope to drive rim to floor (deg)", design_doc=31.6, measured=round(DRIVE_DOWN_ANY, 1)),
    dict(item="Floor in reach of the Dock, baseline (%)", design_doc=f"{DESIGN_DOC['reach_pct']:.0f} (straight-line estimate)",
         measured=round(float(base.floor_in_reach_pct), 0)),
    dict(item="Floor in reach, with heater units (%)", design_doc=f"{DESIGN_DOC['reach_pct_heater']:.0f} (straight-line estimate)",
         measured=round(float(heat.floor_in_reach_pct), 0)),
    dict(item="Reachable floor with direct radio to the Dock (%)", design_doc="not stated", measured=round(RADIO_LOS_PCT, 0)),
])
display(H)

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else round(float(o), 4)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
summary = clean({
    "settings": {"crop_km": CROP_KM, "grid_m": GRID_M, "safe_max_deg": SAFE_MAX_DEG, "nogo_min_deg": NOGO_MIN_DEG,
                 "er_max_deg": ER_MAX_DEG, "rappel_max_deg": RAPPEL_MAX_DEG, "sunlit_penalty": SUNLIT_PENALTY,
                 "floor_frac": FLOOR_FRAC, "dr_rule": DR_RULE, "er_reach_m": ER_REACH_M, "tether_slack": TETHER_SLACK,
                 "er_ant_m": ER_ANT_M, "dock_ant_m": DOCK_ANT_M,
                 "labels": "safe_max, rappel_max sourced; er_reach from concept-sizing; others placeholder or setting"},
    "measured": {"geometry": geo.to_dict(orient="records"), "hazard_by_zone": B.to_dict(orient="records"),
                 "drive_down_deg_from_dr_site": DRIVE_DOWN_DR, "drive_down_deg_any_rim": DRIVE_DOWN_ANY,
                 "floor_reach": C.to_dict(orient="records"), "tether_routes": D.to_dict(orient="records"),
                 "dock_candidates": E_all.drop(columns="rc").to_dict(orient="records"),
                 "dock_reach": E.to_dict(orient="records"), "radio_los_to_dock_pct": RADIO_LOS_PCT,
                 "design_check": H.to_dict(orient="records"),
                 "dr_xy_m": list(dr_xy), "dock_xy_m": list(xy_of(*DOCK))},
    # Inputs read by ../concept-sizing/sizing.py
    "design_inputs": {"route_km": WALL_KM, "map_length_km": float(D.iloc[0].map_length_km), "drop_km": km(float(D.iloc[0].drop_m)),
                      "max_slope": float(D.iloc[0].steepest_5m_deg), "shadow_frac": float(D.iloc[0].in_shadow_pct) / 100,
                      "floor_diam_km": 2 * km(math.sqrt(floor_area / math.pi)), "floor_area_km2": floor_area / 1e6,
                      "floor_drivable": float(drv_floor.sum() / floor.sum()),
                      "floor_run_km": km(ENTRY_LINK_M) + FLOOR_RUN_KM,
                      "floor_in_reach_frac": {k: float(v) / 100 for k, v in zip(E.case, E.floor_in_reach_pct)}},
    "data_citations": ["Barker et al. 2021, PSS 203, 105119, doi:10.1016/j.pss.2020.105119",
                       "Barker et al. 2023, PSJ 4, 183, doi:10.3847/PSJ/acf3e1"],
})
(RESULTS / "m4_summary.json").write_text(json.dumps(summary, indent=2))
print("Wrote", RESULTS / "m4_summary.json")